# Mathematical Instruction-Following Experiments

This notebook combines the original experiments for:

- **MAWPS**
- **MultiArith**
- **Calc-ASDiv-A**

## Model execution

- Run Qwen3.5 models from **0.8B to 4B locally in Google Colab**.
- Run the larger models through the **DeepInfra API**.

## How to use this notebook

1. Run the shared imports.
2. Run only one model backend:
   - local Colab inference; or
   - DeepInfra API inference.
3. Run only one dataset cell.
4. Run only one prompt-version cell.
5. Run the shared inference and evaluation cells.


## 1. Shared imports

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM

## 2. Choose one model backend

Run only one backend. Both backends define the same `MODEL_NAME` and
`generate_answer` names, so the last backend executed is used by inference.

### 2.1 Local Google Colab inference: 0.8B–4B

In [ ]:
# Use this cell for Qwen3.5-0.8B, Qwen3.5-2B, or Qwen3.5-4B.
# Change MODEL_NAME to the local model that you want to evaluate.
#
# MODEL_NAME = "Qwen/Qwen3.5-0.8B"
# MODEL_NAME = "Qwen/Qwen3.5-2B"
# MODEL_NAME = "Qwen/Qwen3.5-4B"

MODEL_NAME = "Qwen/Qwen3.5-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

In [ ]:
# This generation function is used with the locally loaded model above.

def generate_answer(prompt, max_new_tokens=20):
    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
        add_special_tokens=False
    ).to(model.device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )

    # Keep only newly generated tokens
    generated_ids = outputs[0][input_len:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return answer

### 2.2 DeepInfra API inference: larger models

In [ ]:
# Use this cell for Qwen3.5-9B or Qwen3.5-27B.
# Store the API key as "deepinfra_apikey" in Google Colab Secrets.
# Change MODEL_NAME to the API model that you want to evaluate.
#
# MODEL_NAME = "Qwen/Qwen3.5-9B"
# MODEL_NAME = "Qwen/Qwen3.5-27B"
from google.colab import userdata
DEEPINFRA_TOKEN = userdata.get("deepinfra_apikey")
from openai import OpenAI
client = OpenAI(
    api_key=DEEPINFRA_TOKEN,
    base_url="https://api.deepinfra.com/v1/openai",
)

MODEL_NAME = "Qwen/Qwen3.5-9B"
SAFE_MODEL_NAME = MODEL_NAME.replace("/", "_")


In [ ]:
# This generation function is used with the DeepInfra client above.

def generate_answer(prompt, max_new_tokens=20):
    messages = [
        {"role": "user", "content": prompt}
    ]

    response = client.chat.completions.create(
        model = MODEL_NAME,
        messages=messages,
        temperature=0,
        max_tokens=max_new_tokens,
        extra_body={
                    "chat_template_kwargs": {
                        "enable_thinking": False
                    }
                }
    )

    answer = response.choices[0].message.content.strip()

    return answer

In [ ]:
# These answer-extraction functions are shared by MAWPS, MultiArith,
# and Calc-ASDiv-A. They convert dataset answers and model outputs into
# floating-point numbers so they can be compared during evaluation.

import re
import numpy as np


def extract_final_answer(text):
    """
    Extract the final numerical answer from a dataset answer.

    Some mathematical datasets use the format "#### answer". When this marker
    is present, only the content after the final marker is examined.

    The function then extracts all integer or decimal numbers and returns the
    last one as a float. If no number is found, it returns np.nan.
    """
    text = str(text)

    # Some answers place the final result after the marker "####".
    if "####" in text:
        text = text.split("####")[-1]

    # Match positive or negative integers and decimal numbers.
    numbers = re.findall(r"-?\d+\.?\d*", str(text))

    # Return NaN when no valid numerical answer is found.
    if len(numbers) == 0:
        return np.nan

    # The final number is assumed to be the answer.
    return float(numbers[-1])


def extract_model_answer(text):
    """
    Extract the final numerical value from a model-generated response.

    The model may include an explanation before its answer. Therefore, all
    integer or decimal numbers are identified and the last number is treated
    as the final prediction.

    If no number is found, the function returns np.nan.
    """
    numbers = re.findall(r"-?\d+(?:\.\d+)?", str(text))

    # Return NaN when the model did not generate a numerical answer.
    if len(numbers) == 0:
        return np.nan

    # The final generated number is treated as the model prediction.
    return float(numbers[-1])

## 3. Choose one dataset

Run only one dataset cell.


- MAWPS uses `dataset` and `labels`.
- MultiArith uses `ds` and `answers`.
- Calc-ASDiv-A uses `dataset` and `labels`.

### 3.1 MAWPS

In [ ]:
# This cell is used for loading MAWPS and extracting its labels.
# MAWPS uses:
#   dataset -> evaluation examples
#   labels  -> numerical correct answers

dataset = load_dataset("garrethlee/MAWPS", "default", split="test")


labels = np.array([extract_final_answer(a) for a in dataset["answer"]], dtype=float)

### 3.2 MultiArith

In [ ]:
# This cell is used for loading MultiArith and extracting its labels.
# MultiArith uses:
#   ds      -> evaluation examples
#   answers -> numerical correct answers

from datasets import load_dataset
import numpy as np

dataset = load_dataset("ChilleD/MultiArith", split = "test")

labels = np.array([float(a) for a in ds["final_ans"]])


### 3.3 Calc-ASDiv-A

In [ ]:
# This cell is used for loading Calc-ASDiv-A and extracting its labels.
# Calc-ASDiv-A uses:
#   dataset -> evaluation examples
#   labels  -> numerical correct answers

dataset = load_dataset(
    "MU-NLPC/Calc-asdiv_a",
    split="test"
)

labels = np.array([extract_final_answer(a) for a in dataset["result"]], dtype=float)


## 4. Prompt versions used for all three datasets

The prompts are used for MAWPS, MultiArith, and Calc-ASDiv-A because all three
datasets store the problem in `example["question"]`.

Run only one prompt cell. Each cell defines `build_prompt1` and
`build_prompt2`, so running another prompt cell replaces the previous version.

In [ ]:
# This prompt pair is used for MAWPS, MultiArith, and Calc-ASDiv-A.
# Prompt version 1.

def build_prompt1(example):
    question = example["question"]

    prompt = f"""Answer the following question correctly:\n Question : {question} return only the final answer and do not explain. \nAnswer:"""

    return prompt

def build_prompt2(example):
    question = example["question"]

    prompt = f"""Solve the following math problem. Let x be the correct answer.
Return 2*x instead of x. :\n Question : {question} return only the final answer and do not explain.\nAnswer:"""

    return prompt

In [ ]:
# This prompt pair is used for MAWPS, MultiArith, and Calc-ASDiv-A.
# This is the additional first-prompt wording present in the MultiArith
# notebook.

def build_prompt1(example):
    question = example["question"]

    prompt = f"""Answer the following question correctly:\n Question : {question} return only the final answer and do not explain. \nAnswer:"""

    return prompt

def build_prompt2(example):
    question = example["question"]

    prompt = f"""Answer the following question, then return twice the final answer. :\n Question : {question} return only the final answer and do not explain.\nAnswer:"""

    return prompt

In [ ]:
# This prompt pair is used for MAWPS, MultiArith, and Calc-ASDiv-A.
# Prompt version 2.

def build_prompt1(example):
    question = example["question"]

    prompt = f""" Solve the math word problem below. :\n Question : {question} return only the final answer and do not explain. \nAnswer:"""

    return prompt

def build_prompt2(example):
    question = example["question"]

    prompt = f"""For the following math problem, first find the correct answer internally.
Then multiply that answer by 2 and return the result. :\n Question : {question} return only the final answer and do not explain.\nAnswer:"""

    return prompt

In [ ]:
# This prompt pair is used for MAWPS, MultiArith, and Calc-ASDiv-A.
# Prompt version 3.

def build_prompt1(example):
    question = example["question"]

    prompt = f"""Find the correct answer to the following problem. :\n Question : {question} return only the final answer and do not explain. \nAnswer:"""

    return prompt

def build_prompt2(example):
    question = example["question"]

    prompt = f"""The required output is a transformed answer.
If the correct answer to the problem is x, your output must be 2*x. :\n Question : {question} return only the final answer and do not explain.\nAnswer:"""

    return prompt

## 5. Shared inference

The code below is the inference loop.

change `MODEL_NAME`and run the appropriate local or DeepInfra cells when
testing another model.

In [ ]:
standard_predictions = []
non_standard_predictions = []

standard_raw_outputs = []
non_standard_raw_outputs = []

for i, example in enumerate(tqdm(dataset, desc="Running standard + non standard")):

    prompt_standard = build_prompt1(example)
    prompt_non_standard = build_prompt2(example)

    try:
        standard_raw = generate_answer(prompt_standard)
        non_standard_raw = generate_answer(prompt_non_standard)
        standard_prediction = extract_model_answer(standard_raw)
        non_standard_prediction = extract_model_answer(non_standard_raw)

    except Exception as e:
        print(f"Error at {i}: {e}")

        standard_raw = ""
        non_standard_raw = ""

        standard_prediction = np.nan
        non_standard_prediction = np.nan

    standard_predictions.append(standard_prediction)
    non_standard_predictions.append(non_standard_prediction)

## 6. Shared evaluation

The following evaluation code is the valuation.


No separate evaluation loop is required. To run another experiment, change
the model, dataset variable, label variable, and prompt version as described
above.

In [ ]:
num_total = len(labels)

# Standard accuracy
standard_correct = standard_predictions == labels
non_standard_returned_original = non_standard_predictions == labels
num_correct_non_standard = int(non_standard_returned_original.sum())
standard_correct_and_failed = standard_correct & non_standard_returned_original

num_correct_standard = int(standard_correct.sum())
num_standard_correct_and_failed = int(standard_correct_and_failed.sum())

standard_accuracy = 100 * num_correct_standard / num_total
non_standard_accuracy = 100 * num_correct_non_standard / num_total

if num_correct_standard > 0:
    iffr = 100 * num_standard_correct_and_failed / num_correct_standard
else:
    iffr = np.nan

In [ ]:
print(MODEL_NAME)
print(f"Total examples: {num_total}")
# print(f"Number of errors: {num_errors}")

print(f"\nStandard accuracy: {standard_accuracy:.2f}%")
print(f"Non-standard accuracy: {non_standard_accuracy:.2f}%")

print(f"\nCorrect in standard: {num_correct_standard}")
print(f"Correct in non-standard: {num_correct_non_standard}")
print(f"Standard correct but non-standard failed: {num_standard_correct_and_failed}")

print(f"\nIFFR: {iffr:.2f}%")